# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. {'A': 2, 'B': 1}. Мы считаем количество букв, на первом шаге для 'A' берется 0 + 1 = 1, для 'B' — 0 + 1 = 1. Когда снова встречаем 'A', метод .get('A', 0) достает 1 и прибавляет к ней еще одну 1 + 1 = 2
2. [0,1,1] . Списки могут менятся, мы вызвали функцию дважы и дважды добавилось единица к 0
3. 2 . В списке убираются дупликаты и остаются только уникальные элементы два города и они все унифицированы в нижнем регистре
4. {'ab': 2, 'abc': 3} Второе появление ключа 'ab' просто перезаписало то же самое значение длины 2
5. 5 и ничего. Так как через условие будет написан 5 но оно не возвращает ничего 

In [4]:
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)

{'A': 2, 'B': 1}
[0, 1, 1]
2
{'ab': 2, 'abc': 3}
5
None


In [5]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [6]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [7]:
# База
#1
def to_num(s):
    return float(s.replace(" ", ""))

city_map = {
    "almaty": "Алматы",
    "алматы": "Алматы",
    "astana": "Астана",
    "астана": "Астана",
    "shymkent": "Шымкент",
    "шымкент": "Шымкент",
    "karaganda": "Караганда",
    "караганда": "Караганда",
    "aktobe": "Актобе",
    "актобе": "Актобе",
    "atyrau": "Атырау",
    "атырау": "Атырау",
}

def clean_city(s):
    key = s.strip().lower()
    return city_map.get(key, s.strip().title())

In [8]:
#2
cat_counts = {}

for r in rows:
    cat = r["category"]
    cat_counts[cat] = cat_counts.get(cat, 0) + 1

sorted_cats = sorted(cat_counts.items(), key=lambda item: item[1], reverse=True)
print("Заказов по категориям:")
for cat, count in sorted_cats:
    print(f"  {cat}: {count}")

Заказов по категориям:
  Одежда: 1529
  Электроника: 1185
  Дом: 1105
  Красота: 991
  Книги: 653


In [9]:
#3
cat_revenue = {}
cat_paid_orders = {}

for r in rows:
    if r["status"] == "оплачен":
        cat = r["category"]
        amt = to_num(r["amount"])
        
        cat_revenue[cat] = cat_revenue.get(cat, 0.0) + amt
        cat_paid_orders[cat] = cat_paid_orders.get(cat, 0) + 1

print(f"{'Категория':<15} | {'Выручка':>15} | {'Средний чек':>15}")
print("-" * 51)

for cat in cat_revenue:
    rev = cat_revenue[cat]
    avg_check = round(rev / cat_paid_orders[cat])
    print(f"{cat:<15} | {rev:>15,.0f} | {avg_check:>15,}")

Категория       |         Выручка |     Средний чек
---------------------------------------------------
Одежда          |      18,321,080 |          13,286
Красота         |       5,507,620 |           6,188
Дом             |      10,516,320 |          10,655
Книги           |       2,294,100 |           3,935
Электроника     |      65,270,240 |          61,287


In [10]:
#4
month_counts = {}

for r in rows:
    month = r["order_date"][:7]
    month_counts[month] = month_counts.get(month, 0) + 1

for month, count in sorted(month_counts.items()):
    print(f"Месяц {month}: {count} заказов")

Месяц 2026-09: 2782 заказов
Месяц 2026-10: 2681 заказов


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [11]:
# Стандарт
#5
month_revenue = {}

for r in rows:
    if r["status"] == "оплачен":
        month = r["order_date"][:7]
        amt = to_num(r["amount"])
        month_revenue[month] = month_revenue.get(month, 0.0) + amt

rev_sep = month_revenue["2026-09"]
rev_oct = month_revenue["2026-10"]
diff_percent = ((rev_oct - rev_sep) / rev_sep) * 100

print(f"Выручка за сентябрь: {rev_sep:,.0f}")
print(f"Выручка за октябрь:  {rev_oct:,.0f}")
print(f"Изменение: {diff_percent:.1f}%")

Выручка за сентябрь: 57,489,850
Выручка за октябрь:  44,419,510
Изменение: -22.7%


In [12]:
#6
city_all = {}
city_canceled = {}

for r in rows:
    city = clean_city(r["city"])
    city_all[city] = city_all.get(city, 0) + 1
    
    if r["status"] == "отменён":
        city_canceled[city] = city_canceled.get(city, 0) + 1

print("Доля отмен по городам:")
for city in sorted(city_all.keys()):
    total = city_all[city]
    canceled = city_canceled.get(city, 0)
    rate = (canceled / total) * 100
    print(f"  {city}: {rate:.1f}%")

Доля отмен по городам:
  Актобе: 10.3%
  Алматы: 7.9%
  Астана: 8.8%
  Атырау: 7.5%
  Караганда: 8.9%
  Шымкент: 20.4%


In [13]:
#7
customer_revenue = {}

for r in rows:
    if r["status"] == "оплачен":
        cust = r["customer_id"]
        amt = to_num(r["amount"])
        customer_revenue[cust] = customer_revenue.get(cust, 0.0) + amt

top5 = sorted(customer_revenue.items(), key=lambda x: x[1], reverse=True)[:5]

print("Топ-5 клиентов:")
for cust, rev in top5:
    print(f"  {cust}: {rev:,.0f}")

Топ-5 клиентов:
  TEST: 29,999,970
  C01627: 257,180
  C00485: 215,920
  C01338: 192,080
  C02122: 191,670


На первом месте находится клиент с ID TEST с суммой почти 30 млн тенге. Это тестовый аккаунт разработчиков с тремя фиктивными заказами по 9 999 990 ₸, который искажает реальную финансовую отчетность и должен быть удален из анализа.

In [14]:
#8
city_clients = {}

for r in rows:
    city = clean_city(r["city"])
    cust = r["customer_id"]
    city_clients.setdefault(city, set()).add(cust)

print("Уникальных покупателей по городам:")
for city, clients in sorted(city_clients.items()):
    print(f"  {city}: {len(clients)}")

Уникальных покупателей по городам:
  Актобе: 358
  Алматы: 1351
  Астана: 981
  Атырау: 138
  Караганда: 451
  Шымкент: 655


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [15]:
# Челлендж